# 05 · Backbone Benchmark: 6 Models × 5 Folds

**Goal:** choose the backbone for DFU-LiteNet **by evidence**, not by assumption.
Every model is trained and tested with the **same** leakage-free folds, training recipe, augmentation and threshold rule.
The recipe was validated in notebook 04.

| Group | Models |
|---|---|
| Lightweight | MobileNetV3-Small, MobileNetV3-Large, MobileNetV2, EfficientNet-B0 |
| Heavyweight reference | ResNet50, DenseNet121 |
| Baseline | Small CNN trained from scratch (shows the value of ImageNet pretraining) |

**Protocol for each model and each run k = 0..4:** test = fold k, validation = fold (k+1) mod 5, training = the other 3 folds.

**Fairness:** the recipe was validated on MobileNetV3-Large (notebook 04), so it might suit that model best. To remove this "home advantage", **every model tries two fine-tuning learning rates (1e-5 and 1e-4)** in every fold, and keeps whichever gives the lower **validation** loss. The rule is the same for all models, and the test fold is never used to choose.

**Selection rule (fixed before seeing results):** the chosen backbone is the **lightest model** (fewest parameters) whose mean F1 is within one standard deviation of the best model's mean F1. Ties go to higher sensitivity.

**Sensitivity check:** fold 0 was looked at while validating the pipeline in notebook 04. The results and the selection are therefore also reported **without fold 0** (folds 1-4), to show that fold 0 did not bias the conclusion.

**Accelerator:** GPU T4 x2 or P100 · **Internet:** On (to download pretrained weights).
Results are written to `results_per_fold.csv` after **every** run, and finished runs are skipped if a cell is re-run.

**Why each run is a separate process (v2):** the first version crashed after 18 runs ("Kernel died"), because memory builds up when many TensorFlow models are trained one after another in one process. Now every (model, fold) run starts a fresh Python process that exits when done, so memory cannot accumulate. **The training code itself is unchanged.**

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

# ---------------- settings (identical for every model) ----------------
CONFIG = {
    "MODELS": ["MobileNetV3Small", "MobileNetV3Large", "MobileNetV2", "EfficientNetB0", "ResNet50", "DenseNet121"],
    "RUN_SCRATCH": True,              # also train the from-scratch CNN baseline
    "FOLDS": [0, 1, 2, 3, 4],
    "IMG_SIZE": 224, "BATCH": 32,
    "EPOCHS_HEAD": 5,                 # phase 1: backbone frozen
    "EPOCHS_FT": 30,                  # phase 2: fine-tune (BatchNorm stays frozen)
    "LR_HEAD": 1e-3,
    "LR_FT_OPTIONS": [1e-5, 1e-4],    # each model keeps whichever gives lower validation loss
    "PATIENCE": 7,
    "SCRATCH_EPOCHS": 60, "SCRATCH_LR": 1e-3, "SCRATCH_PATIENCE": 10,
    "SEED": 42,
}
# -----------------------------------------------------------------------
json.dump(CONFIG, open("config.json", "w"), indent=2)   # read by train_one.py
print(json.dumps(CONFIG, indent=2))

## 1. Data
Only `keep == True` rows (one file per exact copy): 713 unique images. Each run loads them itself.

## 2. Models and their preprocessing
Each pretrained model expects pixels in its **own** format. Using the wrong one would quietly lower its score and make the comparison unfair.

| Model | Expected input |
|---|---|
| MobileNetV3-Small/Large, EfficientNet-B0 | raw 0-255 (preprocessing built into the model) |
| MobileNetV2 | scaled to [-1, 1] |
| ResNet50 | BGR channel order, ImageNet mean subtracted ("caffe" mode) |
| DenseNet121 | scaled to [0, 1], then normalised by ImageNet mean/std ("torch" mode) |

## 3. The training script
This cell only **writes** `train_one.py` (it does not train). The code is the same as notebook 04 plus the fair learning-rate choice. Read it to see exactly what every run does.

In [ ]:
%%writefile train_one.py
# One training run: python train_one.py <model> <test_fold>
# Same code as notebook 05 v1; settings come from config.json written by the cell above.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import gc, json, os, random, sys, time, warnings
warnings.filterwarnings("ignore", message=".*shuffle=True.*")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from PIL import Image
from scipy.stats import binomtest
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

PATCHES = next(Path("/kaggle/input").rglob("Patches"))
SPLITS = next(Path("/kaggle/input").rglob("splits.csv"))


globals().update(json.load(open("config.json")))

# ---------------- data ----------------
splits = pd.read_csv(SPLITS)
data = splits[splits.keep].reset_index(drop=True)

def load(row):
    im = Image.open(PATCHES / row["class"] / row.file).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

X = np.stack([load(r) for _, r in data.iterrows()]).astype(np.float32)   # raw 0-255 pixels
y = data.label.values.astype(np.float32)                                 # 1 = ulcer, 0 = normal
fold_of = data.fold.values

# ---------------- models and preprocessing ----------------
A = keras.applications
identity = lambda t: t
SPECS = {   # name: (constructor, preprocessing function)
    "MobileNetV3Small": (lambda: A.MobileNetV3Small(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True), identity),
    "MobileNetV3Large": (lambda: A.MobileNetV3Large(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True), identity),
    "MobileNetV2":      (lambda: A.MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), A.mobilenet_v2.preprocess_input),
    "EfficientNetB0":   (lambda: A.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), identity),
    "ResNet50":         (lambda: A.ResNet50(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), A.resnet50.preprocess_input),
    "DenseNet121":      (lambda: A.DenseNet121(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)), A.densenet.preprocess_input),
}

def head(x):
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    return layers.Dense(1, activation="sigmoid", name="classifier")(x)

def build_pretrained(name):
    base = SPECS[name][0]()
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    model = keras.Model(inp, head(base(inp, training=False)), name=name)
    return model, base

def build_scratch():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1 / 255)(inp)
    for f in [32, 64, 128, 256]:
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
    return keras.Model(inp, head(x), name="ScratchCNN")

PREP = {name: spec[1] for name, spec in SPECS.items()}
PREP["ScratchCNN"] = identity

# ---------------- one training run ----------------
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.1, fill_mode="reflect"),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="augment")

def make_ds(ids, prep, training):
    ds = tf.data.Dataset.from_tensor_slices((X[ids], y[ids]))
    if training:
        ds = ds.shuffle(len(ids), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH)
    if training:   # augment on raw pixels, then apply the model's own preprocessing
        ds = ds.map(lambda a, b: (prep(augment(a, training=True)), b), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda a, b: (prep(a), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(name="auc")])

def choose_threshold(p, t):
    # maximise balanced accuracy on the validation fold; ties -> middle of the tied range
    grid = np.linspace(0.01, 0.99, 99)
    bal = np.array([((p >= g)[t == 1].mean() + (p < g)[t == 0].mean()) / 2 for g in grid])
    return float(np.median(grid[bal == bal.max()]))

def run_one(name, test_fold):
    set_seed(SEED)
    val_fold = (test_fold + 1) % 5
    tr = np.where((fold_of != test_fold) & (fold_of != val_fold))[0]
    va, te = np.where(fold_of == val_fold)[0], np.where(fold_of == test_fold)[0]
    prep = PREP[name]
    train_ds, val_ds, test_ds = make_ds(tr, prep, True), make_ds(va, prep, False), make_ds(te, prep, False)
    cw = {0: len(tr) / (2 * (y[tr] == 0).sum()), 1: len(tr) / (2 * (y[tr] == 1).sum())}
    t0 = time.time()

    if name == "ScratchCNN":
        model = build_scratch()
        compile_model(model, keras.optimizers.schedules.CosineDecay(SCRATCH_LR, SCRATCH_EPOCHS * len(train_ds)))
        stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=SCRATCH_PATIENCE, restore_best_weights=True)
        h = model.fit(train_ds, validation_data=val_ds, epochs=SCRATCH_EPOCHS, class_weight=cw, callbacks=[stop], verbose=0)
        chosen, epochs, lr_chosen, head_loss = "scratch", len(h.history["loss"]), SCRATCH_LR, np.nan
        lr_losses = {lr: np.nan for lr in LR_FT_OPTIONS}   # keep the same columns as the pretrained models
    else:
        model, base = build_pretrained(name)
        compile_model(model, LR_HEAD)                                         # phase 1: head only
        model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw, verbose=0)
        head_weights, head_loss = model.get_weights(), model.evaluate(val_ds, verbose=0)[0]
        base.trainable = True                                                 # phase 2: fine-tune, BN frozen
        for l in base.layers:
            if isinstance(l, layers.BatchNormalization):
                l.trainable = False
        best = {"val_loss": head_loss, "weights": head_weights, "chosen": "head-only", "lr": None, "epochs": EPOCHS_HEAD}
        lr_losses = {}
        for lr in LR_FT_OPTIONS:                                              # same two options for every model
            set_seed(SEED)
            model.set_weights(head_weights)                                   # every option starts from the same head
            compile_model(model, keras.optimizers.schedules.CosineDecay(lr, EPOCHS_FT * len(train_ds)))
            stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
            h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw, callbacks=[stop], verbose=0)
            lr_losses[lr] = min(h2.history["val_loss"])
            if lr_losses[lr] < best["val_loss"]:
                best = {"val_loss": lr_losses[lr], "weights": model.get_weights(), "chosen": f"fine-tuned",
                        "lr": lr, "epochs": EPOCHS_HEAD + len(h2.history["loss"])}
        model.set_weights(best["weights"])
        chosen, epochs, lr_chosen = best["chosen"], best["epochs"], best["lr"]
    train_min = (time.time() - t0) / 60

    thr = choose_threshold(model.predict(val_ds, verbose=0).ravel(), y[va])
    p = model.predict(test_ds, verbose=0).ravel()
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y[te], pred, labels=[0, 1]).ravel()
    row = {"model": name, "fold": test_fold, "threshold": thr, "chosen": chosen, "lr_ft": lr_chosen, "epochs": epochs,
           "val_loss_head": head_loss, **{f"val_loss_lr{lr:g}": v for lr, v in lr_losses.items()},
           "train_min": round(train_min, 2), "n_test": len(te), "tp": tp, "fn": fn, "tn": tn, "fp": fp,
           "accuracy": accuracy_score(y[te], pred), "precision": precision_score(y[te], pred, zero_division=0),
           "sensitivity": recall_score(y[te], pred), "specificity": tn / (tn + fp),
           "f1": f1_score(y[te], pred), "auc": roc_auc_score(y[te], p)}
    preds = pd.DataFrame({"model": name, "fold": test_fold, "file": data.file[te].values,
                          "class": data["class"][te].values, "label": y[te].astype(int), "prob_ulcer": p, "pred": pred})
    return row, preds, model

# ---------------- size, FLOPs and speed ----------------
def count_flops(model, prep):
    # FLOPs for one 224x224 image, from TensorFlow's graph profiler (multiply-add = 2 FLOPs)
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        fn = tf.function(lambda t: model(prep(t), training=False))
        conc = fn.get_concrete_function(tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32))
        frozen = convert_variables_to_constants_v2(conc)
        B = tf.compat.v1.profiler.ProfileOptionBuilder
        opts = B(B.float_operation()).with_empty_output().build()   # silence the long profiler printout
        info = tf.compat.v1.profiler.profile(graph=frozen.graph, run_meta=tf.compat.v1.RunMetadata(), cmd="op", options=opts)
        return info.total_float_ops
    except Exception as e:
        print("   FLOPs not available:", type(e).__name__)
        return None

def latency_ms(model, prep, device, runs=100):
    with tf.device(device):
        infer = tf.function(lambda t: model(prep(t), training=False))
        x = tf.constant(X[:1])
        for _ in range(10): infer(x).numpy()
        t = time.perf_counter()
        for _ in range(runs): infer(x).numpy()
        return (time.perf_counter() - t) / runs * 1000

def efficiency(model, name):
    prep = PREP[name]
    return {"model": name, "params_M": model.count_params() / 1e6, "size_mb_fp32": model.count_params() * 4 / 1e6,
            "gflops": (lambda f: f / 1e9 if f else None)(count_flops(model, prep)),
            "latency_gpu_ms": latency_ms(model, prep, "/GPU:0") if tf.config.list_physical_devices("GPU") else None,
            "latency_cpu_ms": latency_ms(model, prep, "/CPU:0", runs=30)}


if __name__ == "__main__":
    name, k = sys.argv[1], int(sys.argv[2])
    RES, PRED, EFF = Path("results_per_fold.csv"), Path("test_predictions_all.csv"), Path("efficiency.csv")
    row, preds, model = run_one(name, k)
    if RES.exists():   # always write the same columns in the same order as the existing file
        pd.DataFrame([row]).reindex(columns=pd.read_csv(RES, nrows=0).columns).to_csv(RES, mode="a", header=False, index=False)
    else:
        pd.DataFrame([row]).to_csv(RES, index=False)
    preds.to_csv(PRED, mode="a", header=not PRED.exists(), index=False)
    if not (EFF.exists() and name in set(pd.read_csv(EFF).model)):
        pd.DataFrame([efficiency(model, name)]).to_csv(EFF, mode="a", header=not EFF.exists(), index=False)
    print(f"RESULT {name:17s} fold {k} | acc {row['accuracy']:.3f}  sens {row['sensitivity']:.3f}  spec {row['specificity']:.3f}"
          f"  F1 {row['f1']:.3f}  AUC {row['auc']:.3f} | thr {row['threshold']:.2f} | {row['chosen']:10s} lr {row['lr_ft']}"
          f" | {row['train_min']:.1f} min")

## 4. Run the benchmark
This is the long cell (about 4 hours). A result line is printed after each (model, fold) run finishes, about every 4-8 minutes. Re-running this cell continues where it left off.

In [ ]:
RES, PRED, EFF = Path("results_per_fold.csv"), Path("test_predictions_all.csv"), Path("efficiency.csv")
queue = CONFIG["MODELS"] + (["ScratchCNN"] if CONFIG["RUN_SCRATCH"] else [])
done = set(map(tuple, pd.read_csv(RES)[["model", "fold"]].values)) if RES.exists() else set()
env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2")

t_start = time.time()
for name in queue:
    for k in CONFIG["FOLDS"]:
        if (name, k) in done:
            continue
        for attempt in (1, 2):                     # one automatic retry if a run fails
            r = subprocess.run([sys.executable, "train_one.py", name, str(k)], capture_output=True, text=True, env=env)
            result = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
            if r.returncode == 0 and result:
                print(result[-1][7:] + f" | total {(time.time() - t_start) / 60:.0f} min", flush=True)
                break
            print(f"!! {name} fold {k}: attempt {attempt} failed (exit code {r.returncode}). Last error lines:")
            print("\n".join((r.stderr or r.stdout).splitlines()[-15:]), flush=True)
        else:
            raise RuntimeError(f"{name} fold {k} failed twice - see the error lines above")
print("Benchmark finished.")

## 5. Results: mean ± standard deviation over the 5 folds

In [ ]:
res = pd.read_csv(RES); eff = pd.read_csv(EFF)
order = queue
metrics = ["accuracy", "sensitivity", "specificity", "precision", "f1", "auc"]
agg = res.groupby("model")[metrics].agg(["mean", "std"])
table = pd.DataFrame({m: [f"{100 * agg.loc[n, (m, 'mean')]:.1f} ± {100 * agg.loc[n, (m, 'std')]:.1f}" for n in order]
                      for m in metrics}, index=order)
table = table.join(eff.set_index("model")[["params_M", "size_mb_fp32", "gflops", "latency_gpu_ms", "latency_cpu_ms"]].round(2))
table.to_csv("summary_table.csv")
table

## 6. Pooled results over all 713 test predictions
Each image is tested exactly once (in its own fold), so pooling gives one confusion matrix per model.

In [ ]:
pred_all = pd.read_csv(PRED)
pooled = []
for n in order:
    d = pred_all[pred_all.model == n]
    tn, fp, fn, tp = confusion_matrix(d.label, d.pred, labels=[0, 1]).ravel()
    pooled.append({"model": n, "TP": tp, "FN (missed ulcers)": fn, "TN": tn, "FP (false alarms)": fp,
                   "accuracy": (tp + tn) / len(d), "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                   "auc_pooled": roc_auc_score(d.label, d.prob_ulcer)})
pooled = pd.DataFrame(pooled).set_index("model")
pooled.round(4)

## 7. Accuracy vs. cost and backbone selection

In [ ]:
def summarise(r):
    a = r.groupby("model")[metrics].agg(["mean", "std"])
    return a.xs("mean", axis=1, level=1).join(a.xs("std", axis=1, level=1).add_suffix("_std")).join(eff.set_index("model"))

def select(summary):
    # The pre-registered rule: lightest model whose mean F1 >= best mean F1 - best std
    c = summary.drop(index="ScratchCNN", errors="ignore")
    b = c.f1.idxmax()
    elig = c[c.f1 >= c.loc[b, "f1"] - c.loc[b, "f1_std"]]
    sel = elig.sort_values(["params_M", "sensitivity"], ascending=[True, False]).index[0]
    return sel, b, elig, c

s = summarise(res)
selected, best, eligible, cands = select(s)
pareto = [n for n in cands.index if not ((cands.params_M <= cands.loc[n, "params_M"]) & (cands.f1 > cands.loc[n, "f1"])).any()]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
for n in s.index:
    c = "#b22222" if n == selected else ("#1f3b63" if n in pareto else "#888888")
    ax[0].errorbar(s.loc[n, "params_M"], 100 * s.loc[n, "f1"], yerr=100 * s.loc[n, "f1_std"], fmt="o", color=c, capsize=4, ms=8)
    ax[0].annotate(n, (s.loc[n, "params_M"], 100 * s.loc[n, "f1"]), textcoords="offset points", xytext=(6, 6), fontsize=9)
    if s.loc[n, "latency_cpu_ms"] == s.loc[n, "latency_cpu_ms"]:
        ax[1].scatter(s.loc[n, "latency_cpu_ms"], 100 * s.loc[n, "f1"], color=c, s=60)
        ax[1].annotate(n, (s.loc[n, "latency_cpu_ms"], 100 * s.loc[n, "f1"]), textcoords="offset points", xytext=(6, 6), fontsize=9)
ax[0].set_xscale("log"); ax[0].set_xlabel("Parameters (millions, log scale)"); ax[0].set_ylabel("F1 (%) mean ± std over 5 folds")
ax[0].set_title("Accuracy vs. model size  (red = selected, blue = Pareto front)")
ax[1].set_xlabel("CPU latency per image (ms)"); ax[1].set_ylabel("F1 (%)"); ax[1].set_title("Accuracy vs. CPU speed")
plt.tight_layout(); plt.savefig("accuracy_vs_cost.png", dpi=130); plt.show()

print(f"Best mean F1: {best} ({100 * cands.loc[best, 'f1']:.1f} ± {100 * cands.loc[best, 'f1_std']:.1f})")
print("Within one std of the best:", list(eligible.index))
print("Pareto front (no other model is both smaller and better):", pareto)
print(f"SELECTED BACKBONE (lightest within one std of best): {selected}")

## 7b. Sensitivity check: results without fold 0
Fold 0 was used to validate the pipeline in notebook 04. If the selected backbone is the same with folds 1-4 only, fold 0 did not bias the conclusion.

In [ ]:
s14 = summarise(res[res.fold != 0])
sel14, best14, elig14, _ = select(s14)
check = pd.DataFrame({"F1 all 5 folds": (100 * s.f1).round(1), "F1 folds 1-4": (100 * s14.f1).round(1),
                      "rank all": s.f1.rank(ascending=False).astype(int), "rank folds 1-4": s14.f1.rank(ascending=False).astype(int)}).loc[order]
print(f"Selected with all folds: {selected} | selected with folds 1-4: {sel14} -> "
      + ("SAME: fold 0 did not change the conclusion" if sel14 == selected else "DIFFERENT: discuss this in the thesis"))
check

### Which fine-tuning learning rate did each model choose?
Chosen on validation loss in every fold. `head-only` means fine-tuning did not lower validation loss.

In [ ]:
lr_table = res.assign(choice=res.apply(lambda r: r.chosen if r.chosen != "fine-tuned" else f"lr {r.lr_ft:g}", axis=1))
lr_table.pivot(index="model", columns="fold", values="choice").loc[order]

## 8. Is the difference real? (McNemar's test on pooled predictions)
For each model, we compare its per-image correctness with the selected backbone's. A p-value below 0.05 means the difference is unlikely to be chance.

In [ ]:
ref = pred_all[pred_all.model == selected].set_index(["file", "class"])
tests = []
for n in order:
    if n == selected: continue
    d = pred_all[pred_all.model == n].set_index(["file", "class"]).loc[ref.index]
    a_ok, b_ok = ref.pred == ref.label, d.pred == d.label
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())   # selected right & other wrong / vice versa
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    tests.append({"model": n, f"{selected} right, other wrong": b, f"other right, {selected} wrong": c, "McNemar p": round(p, 4)})
pd.DataFrame(tests).set_index("model")

In [ ]:
# ROC curves (pooled over folds)
plt.figure(figsize=(6, 5.5))
for n in order:
    d = pred_all[pred_all.model == n]
    fpr, tpr, _ = roc_curve(d.label, d.prob_ulcer)
    plt.plot(fpr, tpr, label=f"{n} (AUC {roc_auc_score(d.label, d.prob_ulcer):.3f})", lw=2 if n == selected else 1)
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("false positive rate"); plt.ylabel("true positive rate")
plt.title("ROC curves, pooled over 5 folds"); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig("roc_pooled.png", dpi=130); plt.show()

json.dump({"selected_backbone": selected, "selected_without_fold0": sel14, "best_f1_model": best, "eligible": list(eligible.index), "pareto": pareto,
           "rule": "lightest model with mean F1 >= best mean F1 - best std"}, open("selection.json", "w"), indent=2)
print(open("selection.json").read())

## What to check
- **The progress lines:** every model should finish all 5 folds. Watch for any fold with very low scores.
- **Section 5:** mean ± std per model. Small standard deviations mean stable results.
- **Section 6:** the number of **missed ulcers** (FN) per model matters most clinically.
- **Section 7:** the chart and the **SELECTED BACKBONE** line. This model becomes the base of DFU-LiteNet in notebook 06.
- **Section 8:** whether differences between models are statistically significant.